# **Sesión 2:** Calibración de Cámara 📷⚙️

## 0. Preparación

In [1]:
import cv2
print("OpenCV should be 4.8.x Current version:", cv2.__version__)
from typing import List
import numpy as np
# import imageio --- IGNORE IF DEPRECATED---
import imageio.v3 as imageio
import copy
import os
import glob
import matplotlib.pyplot as plt


OpenCV should be 4.8.x Current version: 4.8.0


## **Apartado A: Calibración de cámara** (derecha e izquierda)

En este apartado se realiza la calibración de dos cámaras de un sistema estereoscópico. Para ello se trabajará con las imágenes de las carpetas ``left`` y ``right``. En primer lugar se trabajará con la carpeta ``left``. Posteriormente, deberá repetir el proceso con las imágenes en la carpeta ``right``. Ambas carpetas contienen imágenes con las que se calibrarán las cámaras. En ellas aparece el patrón de calibración en diferentes posiciones y orientaciones. Estas imágenes serán los datos de entrada.

Los pasos que deberá seguir para calibrar una cámara son:

1. Defina y ejecute el método para cargar imágenes ``load_images()``.
2. Detecte las esquinas de los patrones usando ``cv2.findChessboardCorners()``. Refine las detecciones con ``cv2.cornerSubPix()``.
3. Compruebe que las detecciones son correctas dibujando los resultados con ``cv2.drawChessboardCorners()``.
4. Defina y ejecute el método ``get_chessboard_points(chessboard_shape, dx, dy)`` que proporcione las coordenadas 3D de las esquinas del patrón. El sistema de referencia utilizado deberá estar anclado al propio patrón.
5. Utilice ``cv2.calibrateCamera`` para obtener los parámetros de calibración para la cámara izquierda.

### **Tarea A.1:** Defina y ejecute el método para cargar imágenes ``load_images()``.

In [2]:
def load_images(filenames: List) -> List:
    return [cv2.imread(filename) for filename in filenames]

In [ ]:
# TODO Build a list containing the paths of all images from the left camera
imgs_path = []
from pathlib import Path
carpeta = Path('data_calibracion')
for archivo in carpeta.iterdir():
    if archivo.is_file() and archivo.name != ".DS_Store":
        imgs_path.append(f"data_calibracion/{archivo.name}")
imgs = load_images(imgs_path)

[array([[[109, 111,  99],
         [109, 111,  99],
         [110, 112, 100],
         ...,
         [235, 212, 190],
         [234, 211, 189],
         [233, 210, 188]],
 
        [[107, 109,  97],
         [107, 109,  97],
         [108, 110,  98],
         ...,
         [235, 212, 190],
         [234, 211, 189],
         [233, 210, 188]],
 
        [[108, 110,  98],
         [108, 110,  98],
         [108, 110,  98],
         ...,
         [236, 213, 191],
         [235, 212, 190],
         [234, 211, 189]],
 
        ...,
 
        [[ 78, 111, 120],
         [ 79, 112, 121],
         [ 79, 112, 121],
         ...,
         [ 31,  51,  52],
         [ 31,  51,  52],
         [ 31,  51,  52]],
 
        [[ 78, 111, 120],
         [ 79, 112, 121],
         [ 79, 112, 121],
         ...,
         [ 31,  51,  52],
         [ 31,  51,  52],
         [ 31,  51,  52]],
 
        [[ 78, 111, 120],
         [ 79, 112, 121],
         [ 79, 112, 121],
         ...,
         [ 31,  51,  52],
  

### **Tarea A.2:** Detecte las esquinas de los patrones usando ``cv2.findChessboardCorners()``. Refine las detecciones con ``cv2.cornerSubPix()``.

In [9]:
# TODO Find corners with cv2.findChessboardCorners()
# Hint: you can use a list comprehension to process all images
pattern_size = (10,7)
corners = [cv2.findChessboardCorners(img,pattern_size) for img in imgs]

In [10]:
corners_copy = copy.deepcopy(corners)
criteria = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 30, 0.01)

# TODO To refine corner detections with cv2.cornerSubPix() you need to input grayscale images. Build a list containing grayscale images.
imgs_gray = [cv2.cvtColor(img,cv2.COLOR_BGR2GRAY) for img in imgs]

corners_refined = [
    cv2.cornerSubPix(imgs_gray[i], corners[i][1], [11, 11], [-1, -1], criteria) 
    if corners[i][0] else None 
    for i in range(len(imgs))
]

### **Tarea A.3:** Compruebe que las detecciones son correctas dibujando los resultados con ``cv2.drawChessboardCorners()``

In [11]:
imgs_copy = copy.deepcopy(imgs)

In [12]:
# TODO Use cv2.drawChessboardCorners() to draw the cornes
for i in range(len(imgs_copy)):
    imgs_copy[i] = cv2.drawChessboardCorners(imgs_copy[i], pattern_size, corners_refined[i], patternWasFound = corners[i][0])

In [13]:
# TODO Show images and save when needed

def show_image(img: np.array, img_name: str = "Image"):
    cv2.imshow(img_name,img)
    cv2.waitKey(1000)
    cv2.destroyAllWindows()

# TODO Complete the method, use every argument
def write_image(output_folder: str, img_name: str, img: np.array):
    img_path = os.path.join(output_folder,img_name)
    cv2.imwrite(img_path,img)

# for i in range(len(imgs_copy)):
#     show_image(imgs_copy[i])
#     write_image("../data2/left",f"Imagen{i}.png",imgs_copy[i])



### **Tarea A.4:** Defina y ejecute el método ``get_chessboard_points(chessboard_shape, dx, dy)``

In [14]:
# TODO 
def get_chessboard_points(chessboard_shape, dx, dy):
    cols, rows = chessboard_shape  
    points = []
    for r in range(rows):
        for c in range(cols):
            points.append([c * dx, r * dy, 0])
    return np.array(points, dtype=np.float32)

In [15]:
# TODO You need the points for every image, not just one (consider a list comprehension)
dx = dy = 15
chessboard_points = get_chessboard_points(pattern_size,dx,dy)

### **Tarea A.5:** Utilice ``cv2.calibrateCamera()`` para obtener los parámetros de calibración para la cámara izquierda

In [16]:
# Filter data and get only those with adequate detections
valid_corners = [corners_refined[i] for i in range(len(imgs)) if corners[i][0]]
objpoints = [chessboard_points for _ in valid_corners]
# Convert list to numpy array with np.float32 elements
valid_corners = np.array(valid_corners,dtype=np.float32)

In [17]:
# TODO
img_size = imgs_gray[0].shape[::-1]
rms, intrinsics, dist_coeffs, rvecs, tvecs = cv2.calibrateCamera(objpoints,valid_corners,img_size,None,None)

# Obtain extrinsics
extrinsics = list(map(lambda rvec, tvec: np.hstack((cv2.Rodrigues(rvec)[0], tvec)), rvecs, tvecs))

In [18]:
# Print outputs
print("Intrinsics:\n", intrinsics)
print("Distortion coefficients:\n", dist_coeffs)
print("Root mean squared reprojection error:\n", rms)

Intrinsics:
 [[1.54483448e+03 0.00000000e+00 7.70971644e+02]
 [0.00000000e+00 1.55631299e+03 1.01863795e+03]
 [0.00000000e+00 0.00000000e+00 1.00000000e+00]]
Distortion coefficients:
 [[ 3.51161008e-01 -2.71866987e+00 -1.30635836e-02 -7.18590963e-04
   7.80665151e+00]]
Root mean squared reprojection error:
 0.6763923535378316
